In [1]:
import imageio

In [2]:
img_arr = imageio.imread('../../data/p1ch4/image-dog/bobby.jpg')
img_arr.shape

/var/folders/97/f9tvwl412dq2pq6wwlstvskh0000gp/T/ipykernel_83639/2364538550.py:1: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  img_arr = imageio.imread('../../data/p1ch4/image-dog/bobby.jpg')


(720, 1280, 3)

In [3]:
import imageio.v2 as imageio
img_arr = imageio.imread('../../data/p1ch4/image-dog/bobby.jpg')
img_arr.shape

(720, 1280, 3)

Note that the use of imageio is for a light exploration of image data types. The authors recommend TorchVision for a more typical default choice.

In [4]:
# imageio produces a numpy array of dimensions height x width x channel
import torch
img = torch.from_numpy(img_arr)
# Torch expects image formatted via channel x height x width
out = img.permute(2,0,1) # permutes dimensions as specified via dimension index order|

In [5]:
out.shape

torch.Size([3, 720, 1280])

Note that because of storage methods, out points to the same storage object as the numpy array, but with a newly specified size, stride, and offset. Changing anything in img will therefore change what out points to.

In [6]:
# pre-allocated a tensor to hold a batch of image data
batch_size = 3
# reshape images (to do later) and use 8 bit integer representation for pixel intensities
# 8 bit is standard for most cameras
batch = torch.zeros(batch_size, 3, 256, 256, dtype=torch.uint8) 

In [14]:
import os 

data_dir = '../../data/p1ch4/image-cats/'
filenames = [name for name in os.listdir(data_dir)
             if os.path.splitext(name)[-1] == '.png'] # split via extension and check its png
for i, filename in enumerate(filenames):
    # load file via imagio (as numpy array)
    img_arr = imageio.imread(os.path.join(data_dir, filename))
    # convert to pytorch tensor (points to same img_arr object)
    img_t = torch.from_numpy(img_arr)
    # reformate for CxHxW format
    img_t = img_t.permute(2,0,1) # only methods ending with _ will perform operation in place
    # keep only the first three channels (color) as some images have other channels included
    img_t = img_t[:3]
    # from a quick check (deleted from the notebook)
    # the below copies the data into batch, so now
    # we are dealing with a different stored object
    batch[i] = img_t

In [15]:
# one method of normalization

# convert to float (from int8 which lies in [0,255])
batch = batch.float()

# normalize to [0,1] via dividing by maximum
batch /= 255.0

In [21]:
# alternatively, scale via mean and variance 
# My guess here is that some images may be too squashed
# in range if we do max normalization. Mean and variance
# would help all values range over the new domain

n_channels = batch.shape[1]
for c in range(n_channels):
    # averaging over all values for all data but slicing for one channel at a time
    # in other words, average all pixel values across all images on red channel,
    # then green channel then blue
    mean = torch.mean(batch[:,c]) 
    std = torch.std(batch[:,c])
    # replace the data with normalized data
    batch[:,c] = (batch[:,c] - mean)/std